# Adding Your Own Model to DeepBullwhip\n\nThis notebook shows the 3-step pattern for extending DeepBullwhip:\n1. Subclass the appropriate ABC\n2. Add the `@register` decorator\n3. Run the benchmark\n\nWe'll create a custom "panic ordering" policy that doubles orders when inventory drops below zero.

## Step 1: Define the Custom Policy

In [ ]:
import numpy as np
from scipy import stats
from deepbullwhip.policy.base import OrderingPolicy
from deepbullwhip.registry import register


@register("policy", "panic_ordering")
class PanicOrderingPolicy(OrderingPolicy):
    """Doubles the OUT order when inventory position is negative.
    
    This models a common real-world behavior where managers panic-order
    when they see stockouts, which amplifies the bullwhip effect.
    """
    
    def __init__(self, lead_time: int, service_level: float = 0.95,
                 panic_multiplier: float = 2.0):
        self.lead_time = lead_time
        self.service_level = service_level
        self.z_alpha = stats.norm.ppf(service_level)
        self.panic_multiplier = panic_multiplier
    
    def compute_order(self, inventory_position, forecast_mean, forecast_std):
        S = (self.lead_time + 1) * forecast_mean + (
            self.z_alpha * forecast_std * np.sqrt(self.lead_time + 1)
        )
        order = max(0.0, S - inventory_position)
        
        # Panic: double the order if inventory position is negative
        if inventory_position < 0:
            order *= self.panic_multiplier
        
        return order


print("PanicOrderingPolicy registered successfully!")

## Step 2: Verify Registration

In [ ]:
from deepbullwhip import list_registered

print("Registered policies:", list_registered("policy"))

## Step 3: Run the Benchmark\n\nNow we can use our custom policy alongside built-in policies.

In [ ]:
from deepbullwhip.benchmark import BenchmarkRunner

runner = BenchmarkRunner(
    chain_config="consumer_2tier",
    demand="semiconductor_ar1",
    T=156,
    N=20,
    seed=42,
)

results = runner.run(
    policies=[
        "order_up_to",
        ("proportional_out", {"alpha": 0.5}),
        "panic_ordering",  # Our custom policy!
    ],
    metrics=["BWR", "FILL_RATE", "TC"],
)

pivot = results.pivot_table(
    index=["policy", "echelon"],
    columns="metric",
    values="value",
    aggfunc="mean",
)
print(pivot.to_string(float_format="%.3f"))

## Results\n\nAs expected, the panic ordering policy amplifies the bullwhip effect significantly compared to the standard OUT and POUT policies. This confirms the intuition that reactive, fear-based ordering behavior worsens supply chain dynamics.\n\nThe 3-step pattern (`subclass` + `@register` + `BenchmarkRunner`) makes it easy to test any custom policy hypothesis against established baselines.